# LangChain Prompt Templates and Output Parsers: PromptTemplate, ChatPromptTemplate, and Pydantic Parsers

> This notebook contains all code examples from the blog post.
> [Read the full post on BotMartz](https://blog.botmartz.com/langchain_2)

**Author:** Soham Sharma · blog.botmartz.com

In [ ]:
!pip install -q langchain langchain-openai langchain-community chromadb

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(
    input_variables=["topic", "audience"],
    template="Write a one-sentence explanation of {topic} for {audience}."
)

# Format the prompt
formatted = template.format(topic="gradient descent", audience="high school students")
print(formatted)
print(type(formatted))

In [ ]:
from langchain_core.prompts import PromptTemplate

# Variables auto-extracted from the template string
template = PromptTemplate.from_template(
    "Summarize this text in {num_sentences} sentences:\n\n{text}"
)

print(template.input_variables)
formatted = template.format(num_sentences=2, text="LangChain is a framework for building LLM-powered applications. It provides abstractions for prompts, chains, agents, and memory.")
print(formatted)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Shorthand tuple syntax: (role, template_string)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}. Always respond in {language}."),
    ("human", "{question}"),
])

messages = prompt.format_messages(
    role="senior Python engineer",
    language="English",
    question="What is the GIL?"
)

for msg in messages:
    print(f"[{msg.type}] {msg.content}")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

# Simulate previous conversation
history = [
    HumanMessage(content="What is PyTorch?"),
    AIMessage(content="PyTorch is an open-source deep learning framework developed by Meta."),
]

messages = prompt.format_messages(
    chat_history=history,
    input="How does it compare to TensorFlow?"
)

for msg in messages:
    print(f"[{msg.type}]: {msg.content[:60]}...")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from datetime import date

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant for {company}. Today's date is {today}. Help with: {task_type}"),
    ("human", "{user_input}"),
])

# Partially fill system-level variables
company_prompt = prompt.partial(
    company="BotMartz",
    today=str(date.today()),
    task_type="technical questions",
)

# Later, fill user-specific variables
messages = company_prompt.format_messages(user_input="What is LCEL?")
print(messages[0].content)
print(messages[1].content)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chain = (
    ChatPromptTemplate.from_template("Name one use case for {technology}.")
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | StrOutputParser()
)

result = chain.invoke({"technology": "vector databases"})
print(type(result))
print(result)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()

chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Always respond with valid JSON only. No prose."),
        ("human", "Return a JSON object with keys 'name', 'founded', and 'known_for' for: {company}"),
    ])
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | parser
)

result = chain.invoke({"company": "OpenAI"})
print(type(result))
print(result)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

class ModelCard(BaseModel):
    name: str = Field(description="Full model name")
    developer: str = Field(description="Organization that created the model")
    release_year: int = Field(description="Year of public release")
    capabilities: List[str] = Field(description="List of 3 key capabilities")
    context_window: int = Field(description="Context window size in tokens")

parser = PydanticOutputParser(pydantic_object=ModelCard)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical AI researcher. Return information as JSON.\n{format_instructions}"),
    ("human", "Create a model card for: {model_name}"),
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0) | parser

result = chain.invoke({"model_name": "GPT-4"})
print(type(result))
print(f"Name: {result.name}")
print(f"Developer: {result.developer}")
print(f"Release year: {result.release_year}")
print(f"Context window: {result.context_window:,} tokens")
print(f"Capabilities: {result.capabilities}")

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class Simple(BaseModel):
    score: int = Field(description="Score from 1-10")
    reason: str = Field(description="One-sentence explanation")

parser = PydanticOutputParser(pydantic_object=Simple)
print(parser.get_format_instructions())

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from langchain.output_parsers import OutputFixingParser
from pydantic import BaseModel

class Rating(BaseModel):
    score: int
    comment: str

base_parser = PydanticOutputParser(pydantic_object=Rating)

# Wrap with the fixing parser
fixing_parser = OutputFixingParser.from_llm(
    parser=base_parser,
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
)

# Simulate malformed output from an LLM
malformed_output = '{"score": "eight", "comment": "Great product"}'

try:
    base_parser.parse(malformed_output)
except Exception as e:
    print(f"Base parser failed: {type(e).__name__}")

# OutputFixingParser calls LLM to repair
fixed = fixing_parser.parse(malformed_output)
print(f"Fixed result: score={fixed.score}, comment={fixed.comment}")
print(f"Type: {type(fixed.score)}")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {role}."),
    ("human", "{question}"),
])

# Wrong: partial a variable that doesn't exist
try:
    bad = prompt.partial(nonexistent_var="value")
    bad.format_messages(role="assistant", question="hi")
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

# Correct
good = prompt.partial(role="Python expert")
result = good.format_messages(question="What is a decorator?")
print(result[0].content)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()

chain = (
    ChatPromptTemplate.from_messages([
        ("system", "{format_instructions}"),
        ("human", "List 5 Python web frameworks."),
    ]).partial(format_instructions=parser.get_format_instructions())
    | ChatOpenAI(model="gpt-4o-mini", temperature=0)
    | parser
)

result = chain.invoke({})
print(type(result))
print(result)